# Weighted Blend поверх ensemble_v2 артефактов

Простое усреднение `0.5 * distil + 0.5 * bert` дало 0.84094 на public LB. Здесь подбираем оптимальные веса по OOF — возможно не 50/50 даёт максимум F1.

**Двумерный grid search:**
- `w ∈ [0.00, 1.00]` шаг 0.05 — вес DistilBERT (BERTweet получает `1 - w`)
- `t ∈ [0.30, 0.60]` шаг 0.01 — threshold

Для каждой пары (w, t) считаем F1 на OOF cleaned train. Выбираем максимум, применяем к test.

**Артефакты, которые нужны** (от `ensemble_v2.ipynb` self-cleaning версии):
- `distil_oof_v2.npy`, `bert_oof_v2.npy` — OOF probs
- `distil_test_v2.npy`, `bert_test_v2.npy` — test probs
- `distil_oof.npy`, `bert_oof.npy` — старые OOFs для воспроизведения cleaning (получения y_clean)

## 1. Импорты

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report

for f in ['distil_oof.npy', 'bert_oof.npy',
          'distil_oof_v2.npy', 'bert_oof_v2.npy',
          'distil_test_v2.npy', 'bert_test_v2.npy']:
    assert os.path.exists(f), f'Missing artifact: {f}'
print('Все артефакты на месте.')

## 2. Загрузка данных и воспроизведение cleaning

Повторяем шаги cleaning из `ensemble_v2.ipynb`, чтобы получить `y_clean` и `df_train_clean` (для overlap fix).

In [ ]:
df_train = pd.read_csv('data/train.csv')
df_test  = pd.read_csv('data/test.csv')

distil_oof_old = np.load('distil_oof.npy')
bert_oof_old   = np.load('bert_oof.npy')
ensemble_oof_old = (distil_oof_old + bert_oof_old) / 2

def majority_label(labels):
    counts = labels.value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        return 1
    return counts.idxmax()

resolved = df_train.groupby('text', sort=False)['target'].agg(majority_label).reset_index()
first_meta = df_train.groupby('text', sort=False)[['keyword', 'location']].first().reset_index()
df_train_dedup = resolved.merge(first_meta, on='text')

first_pos = df_train.reset_index().groupby('text', sort=False)['index'].first()
dedup_oof = ensemble_oof_old[first_pos.loc[df_train_dedup['text']].values]

FLIP_HIGH, FLIP_LOW = 0.90, 0.10
y_dedup = df_train_dedup['target'].values.copy()
y_dedup[(dedup_oof > FLIP_HIGH) & (y_dedup == 0)] = 1
y_dedup[(dedup_oof < FLIP_LOW)  & (y_dedup == 1)] = 0

df_train_clean = df_train_dedup.copy()
df_train_clean['target'] = y_dedup
y_clean = y_dedup

print(f'Cleaned train: {len(df_train_clean)} строк')
print(f'y_clean distribution: {dict(zip(*np.unique(y_clean, return_counts=True)))}')

## 3. Загрузка v2 OOFs и test probs

In [ ]:
distil_oof  = np.load('distil_oof_v2.npy')
bert_oof    = np.load('bert_oof_v2.npy')
distil_test = np.load('distil_test_v2.npy')
bert_test   = np.load('bert_test_v2.npy')

assert len(distil_oof) == len(y_clean), f'OOF/y_clean size mismatch'

print(f'OOF shape: {distil_oof.shape},  test shape: {distil_test.shape}')
print(f'DistilBERT OOF F1@0.5: {f1_score(y_clean, (distil_oof > 0.5).astype(int)):.4f}')
print(f'BERTweet   OOF F1@0.5: {f1_score(y_clean, (bert_oof   > 0.5).astype(int)):.4f}')
print(f'Equal 0.5  OOF F1@0.5: {f1_score(y_clean, ((distil_oof + bert_oof)/2 > 0.5).astype(int)):.4f}')

## 4. 2D grid search: вес × threshold

21 × 31 = 651 оценок. Считается за секунду.

In [ ]:
weights    = np.arange(0.0, 1.01, 0.05)
thresholds = np.arange(0.30, 0.61, 0.01)

f1_grid = np.zeros((len(weights), len(thresholds)))
for i, w in enumerate(weights):
    blend = w * distil_oof + (1 - w) * bert_oof
    for j, t in enumerate(thresholds):
        f1_grid[i, j] = f1_score(y_clean, (blend > t).astype(int))

best_i, best_j = np.unravel_index(np.argmax(f1_grid), f1_grid.shape)
best_w = weights[best_i]
best_t = thresholds[best_j]
best_f1 = f1_grid[best_i, best_j]

print(f'Best blend: w_distil={best_w:.2f}  w_bert={1-best_w:.2f}  threshold={best_t:.2f}')
print(f'Best OOF F1: {best_f1:.4f}')
print(f'(50/50 + 0.50 was: {f1_grid[10, 20]:.4f})')

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(f1_grid, aspect='auto', cmap='viridis', origin='lower',
               extent=[thresholds[0], thresholds[-1], weights[0], weights[-1]])
ax.scatter([best_t], [best_w], color='red', marker='*', s=200, label=f'best F1={best_f1:.4f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Weight of DistilBERT (BERTweet = 1 - w)')
ax.set_title('OOF F1 grid: weight × threshold')
ax.legend()
plt.colorbar(im, ax=ax, label='OOF F1')
plt.tight_layout(); plt.show()

## 5. Apply лучшего blend к test + overlap fix + submission

In [ ]:
test_blend = best_w * distil_test + (1 - best_w) * bert_test
test_preds = (test_blend > best_t).astype(int).tolist()

print(f'До overlap fix → Disaster: {sum(test_preds)}, Not disaster: {len(test_preds) - sum(test_preds)}')

overlap_labels = (
    df_train_clean[df_train_clean['text'].isin(df_test['text'])]
    .set_index('text')['target']
    .to_dict()
)

overridden = 0
for i, text in enumerate(df_test['text']):
    if text in overlap_labels:
        test_preds[i] = int(overlap_labels[text])
        overridden += 1

print(f'Overlap fix: перезаписано {overridden} предсказаний')
print(f'После overlap fix → Disaster: {sum(test_preds)}, Not disaster: {len(test_preds) - sum(test_preds)}')

submission = pd.read_csv('data/sample_submission.csv')
submission['target'] = test_preds
submission.to_csv('submission_weighted_blend.csv', index=False)

print(f'\nsubmission_weighted_blend.csv saved (w_distil={best_w:.2f}, t={best_t:.2f})')
submission.head()